# Week 1 — Game Theory Foundations & Problem Framing
**SoC 2026 | Dynamic Pricing Using Reinforcement Learning**

**Topics Covered:**
- Nash Equilibrium (intuition + formal definition)
- Dominant Strategies & Iterated Elimination
- Prisoner's Dilemma
- Bertrand Competition Model
- Analytical derivation of Nash price
- Calvano et al. (2020) — Paper Summary

**Resources studied this week (as per SOC mentor PDF):**
- TED-Ed: Game Theory — Science of Decision Making (5 min)
- Yale Open Course — Lectures 1 & 3 (Ben Polak)
- Bertrand Competition Explained (EconplusDal)
- Osborne & Rubinstein — Ch. 2 & 3
- Calvano et al. (2020) — Intro + Section 2

## 1. Nash Equilibrium — Theory

A **Nash Equilibrium** is a strategy profile $(s_1^*, s_2^*, ..., s_n^*)$ such that no player can improve their payoff by *unilaterally* deviating.

$$\forall i, \forall s_i : u_i(s_i^*, s_{-i}^*) \geq u_i(s_i, s_{-i}^*)$$

### Key Distinctions:
| Concept | Meaning |
|---|---|
| Nash Equilibrium | No player benefits from unilateral deviation |
| Pareto Optimality | No outcome makes everyone better off |
| Dominant Strategy | Best response *regardless* of what others do |

> ⚠️ **Common Pitfall (Week 1 SOC notes):** Nash Equilibrium ≠ Pareto Optimal. In the Prisoner's Dilemma, both defecting is the NE but NOT Pareto optimal.

## 2. Prisoner's Dilemma

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Payoff matrix: (Player A payoff, Player B payoff)
# Rows: Player A (Cooperate, Defect)
# Cols: Player B (Cooperate, Defect)
payoff_matrix = {
    ('C', 'C'): (3, 3),   # Both cooperate — good for both
    ('C', 'D'): (0, 5),   # A cooperates, B defects — B exploits A
    ('D', 'C'): (5, 0),   # A defects, B cooperates — A exploits B
    ('D', 'D'): (1, 1),   # Both defect — Nash Equilibrium (mutual punishment)
}

print("Prisoner's Dilemma Payoff Matrix")
print(f"{'':12} {'B: Cooperate':>15} {'B: Defect':>15}")
print("-" * 45)
for a_action in ['C', 'D']:
    row = f"A: {'Cooperate' if a_action=='C' else 'Defect  ':10}"
    for b_action in ['C', 'D']:
        p = payoff_matrix[(a_action, b_action)]
        row += f"  ({p[0]}, {p[1]})         "
    print(row)

print("\nNash Equilibrium: (Defect, Defect) → payoff (1,1)")
print("Pareto Optimal  : (Cooperate, Cooperate) → payoff (3,3)")
print("→ NE ≠ Pareto Optimal in this game!")

## 3. Bertrand Competition Model

**Setup:** Two firms, homogeneous product, simultaneous price setting.

**Demand:** $Q_i = a - b P_i + c P_j$

**Profit:** $\pi_i = (P_i - mc) \cdot Q_i$

**Best Response of Firm i:**  
$$P_i^*(P_j) = \frac{a + b \cdot mc + c \cdot P_j}{2b}$$

**Nash Equilibrium Price (both firms symmetric):**
$$P_{Nash} = \frac{a + b \cdot mc}{2b - c}$$

**Collusive Price (joint monopoly):**
$$P_{collude} = \frac{a + b \cdot mc}{2(b - c)}$$

In [ ]:
# Analytical derivation of Nash and Collusive prices
a, b, c, mc = 10.0, 2.0, 0.5, 1.0

p_nash    = (a + b * mc) / (2 * b - c)
p_collude = (a + b * mc) / (2 * (b - c))

print("=" * 45)
print(f"Market Parameters: a={a}, b={b}, c={c}, mc={mc}")
print("=" * 45)
print(f"Nash Equilibrium Price  : {p_nash:.4f}")
print(f"Collusive Price          : {p_collude:.4f}")
print(f"Marginal Cost (floor)    : {mc:.4f}")
print()

# Compute profits at each benchmark
def profit(pi, pj, a=10, b=2, c=0.5, mc=1):
    q = max(a - b * pi + c * pj, 0)
    return (pi - mc) * q

print(f"Profit at Nash price     : {profit(p_nash, p_nash):.4f} per firm")
print(f"Profit at Collude price  : {profit(p_collude, p_collude):.4f} per firm")
print(f"Defection profit (collude vs nash): {profit(p_nash, p_collude):.4f}")

In [ ]:
# Best Response Functions — visualisation
p_range = np.linspace(mc, 8.0, 200)

def best_response(pj, a=10, b=2, c=0.5, mc=1):
    return (a + b * mc + c * pj) / (2 * b)

br_1 = best_response(p_range)    # Firm 1's BR to Firm 2's price
br_2 = best_response(p_range)    # Symmetric — same BR function

plt.figure(figsize=(7, 6))
plt.plot(p_range, br_1, label="Firm 1 Best Response P1*(P2)", color="steelblue")
plt.plot(br_2, p_range, label="Firm 2 Best Response P2*(P1)", color="coral")
plt.axvline(p_nash, color='red',   linestyle='--', label=f"Nash P={p_nash:.2f}")
plt.axhline(p_nash, color='red',   linestyle='--')
plt.scatter([p_nash], [p_nash], color='red', zorder=5, s=100, label="NE")
plt.xlabel("Price Firm 2")
plt.ylabel("Price Firm 1")
plt.title("Bertrand Duopoly — Best Response Functions")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../results/figures/week1_best_response.png", dpi=150)
plt.show()
print("Figure saved → results/figures/week1_best_response.png")

## 4. Profit Landscape

In [ ]:
# 2D profit heatmap for Firm 1 across all (P1, P2) combinations
prices = np.linspace(mc, 8.0, 80)
P1, P2 = np.meshgrid(prices, prices)
PROFIT1 = np.vectorize(profit)(P1, P2)

plt.figure(figsize=(7, 6))
plt.contourf(P1, P2, PROFIT1, levels=30, cmap="RdYlGn")
plt.colorbar(label="Firm 1 Profit")
plt.axvline(p_nash,    color='red',   linestyle='--', linewidth=1.5, label=f"P_Nash={p_nash:.2f}")
plt.axhline(p_nash,    color='red',   linestyle='--', linewidth=1.5)
plt.axvline(p_collude, color='blue',  linestyle='--', linewidth=1.5, label=f"P_Collude={p_collude:.2f}")
plt.axhline(p_collude, color='blue',  linestyle='--', linewidth=1.5)
plt.xlabel("Firm 1 Price (P1)")
plt.ylabel("Firm 2 Price (P2)")
plt.title("Firm 1 Profit Landscape — Bertrand Duopoly")
plt.legend()
plt.tight_layout()
plt.savefig("../results/figures/week1_profit_landscape.png", dpi=150)
plt.show()

## 5. Calvano et al. (2020) — Summary

**Paper:** *Artificial Intelligence, Algorithmic Pricing, and Collusion*  
Calvano, Calzolari, Denicolò, Pastorello — QJE 2020

### Key Findings (Intro + Section 2):
- Q-learning agents placed in a Bertrand duopoly **autonomously learn to collude** without communication
- Agents achieve **supra-competitive profits** — prices converge above Nash, toward the collusive level
- The collusion is **self-sustaining**: agents implicitly punish deviators (like Tit-for-Tat)
- This raises **antitrust concerns** — collusion without explicit coordination

### Market Setup (Section 2):
- Logit demand model (we use linear demand as simplification)
- Discrete action space of prices
- Q-learning with ε-greedy exploration
- Agent state = last k prices of all firms

### Why This Matters for Our Project:
We replicate this setup with a simpler linear demand model to study the same phenomenon.
The Week 4 Q-learning agent will be compared to this paper's findings.

## 6. Week 1 Summary

| Concept | Key Takeaway |
|---|---|
| Nash Equilibrium | Stable — no unilateral deviation helps |
| Bertrand NE | Price = Marginal Cost (competitive outcome) |
| Collusive Price | Above NE — requires coordination or learning |
| Calvano 2020 | RL agents learn to collude autonomously |
| Our Project Goal | Replicate and study this in simpler linear market |

**Next Week:** Build the custom OpenAI Gymnasium pricing environment.